In [41]:
from itertools import combinations
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from sklearn.datasets import make_blobs

jax.config.update('jax_enable_x64', True)
#enables 64 bit float numbers

SEED = 42
SAMPLES = 250

class DataBase:
    """Stores a database consisting of single input feature vectors and their respective labels.
    Includes operations that you can apply to it: 
                - Separate data into train/val/test
                - Create all possible pairs"""
    def __init__(self, X, y):
        self.features = jnp.array(X, dtype=jnp.float64)
        self.labels = jnp.array(y, dtype=jnp.float64)

    def split_raw_data(self):
        xs_train, xs_test, ys_train, ys_test = train_test_split(
            self.features, self.labels, test_size=0.5, random_state=SEED)
        xs_val, xs_test, ys_val, ys_test = train_test_split(
            xs_test, ys_test, test_size=0.5, random_state=SEED)
        return {'train': DataBase(xs_train, ys_train), 'val': DataBase(xs_val, ys_val), 'test': DataBase(xs_test, ys_test)}

    def build_pairs(self):
        def create_pairs(X, y):
            n = len(X)
            pairs, labels = [], []
            for i, j in combinations(range(n), 2):
                pairs.append([X[i].tolist(), X[j].tolist()])
                labels.append(1 if y[i] == y[j] else 0)
            return pairs, labels
        X_pairs, y_pairs = create_pairs(self.features, self.labels)
        return DataBase(X_pairs, y_pairs)
    

class SplittedDb:
    """Stores a dataset splitted into train/val/test"""
    def __init__(self, database):
        sets = database.split_raw_data()
        self.train = sets['train']
        self.val = sets['val']
        self.test = sets['test']


class PairsSet:
    """Stores a dataset that consists of the pairs of
    an original dataset X,y. The pairs are already splitted
    into test/val/test"""
    def __init__(self, X, y):
        self.original = DataBase(X, y)
        self.split = SplittedDb(self.original)
        self.train = self.split.train.build_pairs()
        self.len_train = len(self.train.labels)
        self.val = self.split.val.build_pairs()
        self.test = self.split.test.build_pairs()



def sinus_dataset_generator(samples: int, a: float = 0.8, seed: int = SEED):
    """Code taken from Pablo Rodriguez-Grasa, Yue Ban, and Mikel Sanz
      Neural quantum kernels: Training quantum kernels with quantum 
      neural networks(2025)
       Generates a dataset of two possible classes divided by the decision 
        frontier: x1=-a·sin(π x0) for 'a' a configurable parameter"""
    rng = np.random.default_rng(seed)
    n_per_class = samples // 2
    X, y = [], []
    count = [0, 0]

    while count[0] < n_per_class or count[1] < n_per_class:
        x = rng.uniform(-1.0, 1.0, size=(2,)).astype(np.float32)
        label = int(x[1] < -a * np.sin(np.pi * x[0]))
        if count[label] < n_per_class:
            X.append(x)
            y.append(label)
            count[label] += 1

    X = np.array(X)
    y = np.array(y)
    perm = rng.permutation(len(y))
    return X[perm], y[perm]

X, y = make_blobs(n_samples=SAMPLES, centers=3, n_features=2, cluster_std=1.0, random_state=SEED)
database = DataBase(X,y)
splitted = SplittedDb(database)
dataset = PairsSet(X, y)

In [42]:
from enum import Enum, auto
import pennylane as qml
from itertools import combinations
from dataclasses import dataclass

LAYERS_LIST = [i+1 for i in range(10)]
N_QUBITS = 2


@dataclass # included so that the base values are modifiable
class HyperparametersModel:
    """Hyperparameters selection for training:
                    - max epochs: maximum number of training epochs
                    - patience: epochs without an improvement before early stopping
                    - tolerance: minimum loss improvement to reset patience counter
                    - batch size: size of the batches created in the training
                    - learning rate: learning rate for the ADAM optimizer"""

    def __init__(
        self, max_epochs, patience, tolerance, batch_size, learning_rate
    ):
        self.max_epochs = max_epochs
        self.patience = patience
        self.tolerance = tolerance
        self.batch_size = batch_size
        self.learning_rate = learning_rate

class Architecture(Enum):
    OVERLAP = auto()
    SIMILARITY_FUNCTION = auto()


class EntanglementForm(Enum):
    CIRCULAR = auto()
    LINEAR = auto()
    NONE = auto()


class VariationalForm(Enum):
    TWOLOCAL = auto()
    TREETENSOR = auto()


class EncodingForm(Enum):
    ANGLE = auto()
    ZZ = auto()


class Encoding:
    def __init__(self, form: EncodingForm):
        self.form = form
    def circuit(self, x, n_qubits):
        """Returns a function that applies the encoding to a quantum circuit.
        This method has to be called for each input vector"""
        if self.form is EncodingForm.ANGLE:
            def angle_encoding():
                """Angle encoding: RY(x_i) over each qubit, with RZ(x0*x1) as an
                optional addition if cros_term = True"""
                for i in range(n_qubits):
                    qml.RY(x[i], wires=i)
            return angle_encoding
        
        elif self.form is EncodingForm.ZZ:
            def zz_feature(rescale=True):
                """ZZ feature map (Havlicek et al., 2019),
                  by default the inputs are reescaled to [0,pi]"""
                x_scaled = (x + 1) * (np.pi / 2.0) if rescale else x
                nload = min(len(x_scaled), n_qubits)

                for i in range(nload):
                    qml.Hadamard(i)
                    qml.RZ(2 * x_scaled[i], wires=i)

                for q0, q1 in combinations(range(nload), 2):
                    qml.CNOT(wires=[q0, q1])
                    qml.RZ(2.0 * (np.pi - x_scaled[q1]) * (np.pi - x_scaled[q0]), wires=q1)
                    qml.CNOT(wires=[q0, q1])
            return zz_feature
        else:
            raise ValueError(f'{self.form} encoding not supported')


class Variational:
    def __init__(self, entanglement: EntanglementForm, form: VariationalForm, n_rotations: int):
        self.entanglement = entanglement
        self.form = form
        self.rotations = n_rotations

    def entanglement_block(self, n_qubits):
        """Returns a function that applies the chosen entanglement pattern to a quantum circuit"""
        if self.entanglement is EntanglementForm.CIRCULAR:
            def circular_entanglement():
                """Circular entanglement: CNOT(0,1), CNOT(1,2), ..., CNOT(n-2,n-1),CNOT(n-1,0)"""
                if n_qubits <= 1:
                    return
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
                qml.CNOT(wires=[n_qubits - 1, 0])
            return circular_entanglement
        elif self.entanglement is EntanglementForm.LINEAR:
            def linear_entanglement():
                """Linear entanglement: CNOT(0,1), CNOT(1,2), ..., CNOT(n-2,n-1)"""
                if n_qubits <= 1:
                    return
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
            return linear_entanglement
        elif self.entanglement is EntanglementForm.NONE:
            def no_entanglement():
                return
            return no_entanglement
        else:
            raise ValueError(f'Unknown entanglement {self.entanglement}')

    def circuit(self, layers, theta, n_qubits):
        """Returns a function that applies the variational block to a quantum circuit.
        It takes the layers index which indicates the data reuploading layer this block belongs
        to as input. Theta is an array containing the trainable parameters"""
        entanglement_circuit = self.entanglement_block(n_qubits)
        if self.form is VariationalForm.TWOLOCAL:
            def TwoLocal():
                """Standard Two Local circuit: for each rotation step, RY rotations are applied to all qubits
                followed immediately by the entangling block.
                Theta should be of size [layers, n_qubits, n_rotations]"""
                for rotation in range(self.rotations):
                    for i in range(n_qubits):
                        qml.RY(theta[layers, i, rotation], wires=i)
                    entanglement_circuit()
            return TwoLocal

        elif self.form is VariationalForm.TREETENSOR:
            def TreeTensor():
                """Tree Tensor form applied strictly for 2 qubits"""
                if n_qubits != 2:
                    raise ValueError(f'Tree Tensor only available for 2 qubits')
                for i in range(n_qubits):
                    qml.RY(theta[layers, i, 0], wires=i)
                qml.CNOT(wires=[1, 0])  # control=1, target=0
                qml.RY(theta[layers, 0, 1], wires=0)
            return TreeTensor
        else:
            raise ValueError(f'Unknown variational form {self.form}')


class Configuration:
    """Describes the model: number of qubits, encoding used, variational form used.
    It does not include the number of data reuploading layers since some experiments
    sweep through different possible data reuploading layers for the model."""

    def __init__(self, n_qubits, encoding, variational):
        self.qubits = n_qubits
        self.encoding = encoding
        self.variational = variational


class SQNNModel:
    """Stores all the information required to build an overlap and a similarity 
    function architecture for a similarity problem"""
    def __init__(self, architecture: Architecture, configuration: Configuration):
        self.architecture = architecture
        self.configuration = configuration
        self.dev = qml.device('default.qubit', wires=self.configuration.qubits)
        # to keep track of training results for different
        self.result = {L: [] for L in LAYERS_LIST}
        # number of data reuploading layers applied.

    def create_qnn(self):
        """Build the standard quantum neural network as a callable function"""
        def qnn(x, theta, layers):
            """The circuit performs the encoding block followed by the variational
            block a number layers of times for layers the number of data reuploading repetitions"""
            for layer in range(layers):
                self.configuration.encoding.circuit(x, self.configuration.qubits)()
                self.configuration.variational.circuit(layer, theta, self.configuration.qubits)()
        return qnn

    def qnode_generator(self, layers):
        """Builds the callable function for the similarity measurement
        This similarity measurement will be different for each of the architectures
        that will estimate the similarity between two inputs x1 and x2 in different ways:
        - Similarity function architecture: Applies the quantum neural network over the two inputs
                                            and calculates the similarity as a classical  metric between
                                            the outputs
        - Overlap architecture: Applies the quantum neural network to the inputs x1 and x2 but instead of measuring
                                to obtain an output, the inner product between the two states created is given as the output """
        qnn = self.create_qnn()

        @jax.jit
        def similarity_measurement(x, theta):
            x1 = x[0]
            x2 = x[1]
            if self.architecture is Architecture.OVERLAP:
                def overlap_squared(x1, x2, theta):
                    @qml.qnode(self.dev, interface='jax', diff_method='backprop')
                    def overlap_circuit(x1, x2, theta):
                        qnn(x1, theta, layers)  # creates U(x1,theta)|0>
                        qml.adjoint(qnn)(x2, theta, layers)  # adds U*(x2,theta) to U(x1,theta)|0>
                        return qml.probs(wires=range(self.configuration.qubits))
                    return overlap_circuit(x1, x2, theta)[0]  # returns |<ψ(x2)|ψ(x1)>|^2.
                return overlap_squared(x1, x2, theta)  # returns a function that
                # evaluates |<ψ(x2)|ψ(x1)>|^2.

            elif self.architecture is Architecture.SIMILARITY_FUNCTION:
                def similarity_function(x1, x2, theta):
                    @qml.qnode(self.dev, interface='jax', diff_method='backprop')
                    def qnn_ev(x, theta):
                        qnn(x, theta, layers)
                        return qml.expval(qml.PauliZ(0))
                    pred1 = qnn_ev(x1, theta)  # evaluates U(x1, theta)|0>
                    pred2 = qnn_ev(x2, theta)  # evaluates U(x2, theta)|0>
                    return jnp.exp(-(pred1-pred2)**2)  # measures how far apart from each other
                    # the outputs are
                return similarity_function(x1, x2, theta)
            else:
                raise ValueError(f'Unkown architecture {self.architecture}')
        return similarity_measurement  # returns a function that
        # evaluates exp(-(QNN(x1)-QNN(x2))**2


class QNNModel:
    """Stores all the information required to build a standard quantum neural network
    for a classification problem"""
    def __init__(self, configuration: Configuration):
        self.configuration = configuration
        self.dev = qml.device('default.qubit', wires=self.configuration.qubits)
        # to keep track of training results for different
        self.result = {L: [] for L in LAYERS_LIST}
        # number of data reuploading layers applied.

    def create_qnn(self):
        """Build the standard quantum neural network as a callable function"""
        def qnn(x, theta, layers):
            """The circuit performs the encoding block followed by the variational
            block a number layers of times for layers the number of data reuploading repetitions"""
            for layer in range(layers):
                self.configuration.encoding.circuit(x, self.configuration.qubits)()
                self.configuration.variational.circuit(layer, theta, self.configuration.qubits)()
        return qnn

    def qnode_generator(self, layers):
        qnn = self.create_qnn()
        @jax.jit
        def prediction(x, theta):
            @qml.qnode(self.dev, interface='jax', diff_method='backprop')
            def qnn_ev(x, theta):
                qnn(x, theta, layers)
                return qml.expval(qml.PauliZ(0))
            return qnn_ev(x, theta)[0]
        return prediction
    

    def similarity_generator(self, layers):
        qnn = self.qnode_generator(layers)
        def similarity_measurement(x,theta):
            """Prediction over input pairs"""
            x1 = x[0]
            x2 = x[1]
            cls1 = (qnn(x1, theta)>0)
            cls2 = (qnn(x2, theta)>0)
            return (cls1 == cls2)
        return similarity_measurement

# Predefined model configuration used in the experiments
# The naming procedure is: {encoding}-{variational form}-
#                          {number of rotations}-{entanglement}
configurations = {
    'angle-twolocal-1-ent': Configuration(
        n_qubits=N_QUBITS,
        encoding=Encoding(EncodingForm.ANGLE),
        variational=Variational(entanglement=EntanglementForm.LINEAR,
                                form=VariationalForm.TWOLOCAL, 
                                n_rotations=1)),

    'angle-twolocal-2-ent': Configuration(
        n_qubits=N_QUBITS,
        encoding=Encoding(EncodingForm.ANGLE),
        variational=Variational(entanglement=EntanglementForm.CIRCULAR,
                                form=VariationalForm.TWOLOCAL, 
                                n_rotations=2)),

    # model4 (n_rotations/entanglement does not apply to TreeTensor)
    'angle-treetensor': Configuration(
        n_qubits=N_QUBITS,
        encoding=Encoding(EncodingForm.ANGLE),
        variational=Variational(entanglement=EntanglementForm.NONE,
                                form=VariationalForm.TREETENSOR, 
                                n_rotations=2)),
}



In [44]:
import sys
from circuits import *
from sklearn.metrics import (
    accuracy_score,
    roc_curve,
    roc_auc_score,
    confusion_matrix
)
import time
import numpy as np
import optax
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import cm

# number of repetitions of each experiment over
# different initial parameters each
SEEDS_HYPER = 40
SEEDS_EVALUATE = 10
THRESHOLD = 0.5
SCALING_FACTOR = 0.2 * jnp.pi


class TrainingResult:
    """Stores all the relevant metrics obtained in the training loop"""
    def __init__(self, theta, best_val_loss, final_train_loss, val_losses, train_losses, epochs, total_time):
        self.theta = theta
        self.epochs = epochs
        self.best_val_loss = best_val_loss
        self.time = total_time
        self.val_losses = val_losses
        self.train_losses = train_losses
        self.final_train_loss = final_train_loss

class SeedStorage:
    def __init__(self):
        self.val_loss = []
        self.time = []
        self.epochs = []
        self.accuracy = []
        self.auc = []


class TrainedModel:
    def __init__(self, best_training_result: TrainingResult, over_seeds: SeedStorage):
        self.best = best_training_result

        self.avg_val_over_seeds = np.average(over_seeds.val_loss)
        self.std_val_over_seeds = np.std(over_seeds.val_loss)
        self.avg_time = np.average(over_seeds.time)
        self.std_time = np.std(over_seeds.time)
        self.avg_epochs = np.average(over_seeds.epochs)
        self.std_epochs = np.std(over_seeds.epochs)

class TrainingEvaluation:
    """Stores all the metrics used to evaluate the test dataset for a trained model"""
    def __init__(self, training, accuracy, roc, auc, confusion):
        self.theta = training.theta
        self.epochs = training.epochs
        self.time = training.time
        self.val_losses = training.val_losses
        self.best_val_loss = training.best_val_loss
        self.final_train_loss = training.final_train_loss
        self.train_losses = training.train_losses

        self.accuracy = accuracy
        self.roc = roc
        self.auc = auc
        self.confusion = confusion       

class EvaluatedModel:
     def __init__(self, best_training_evaluation: TrainingEvaluation, over_seeds: SeedStorage):

        self.best = best_training_evaluation
        self.val_over_seeds = over_seeds.val_loss
        self.avg_time = np.average(over_seeds.time)
        self.std_time = np.std(over_seeds.time)
        self.acc_over_seeds = over_seeds.accuracy
        self.auc_over_seeds = over_seeds.auc


def train_model(model, hyperparameters_model, layers, dataset, seeds) -> TrainedModel:
    """Runs the training process over a dataset for a model with layers data reuploading layers and some hyperparameters.
        To perform an statistical evaluation of the model a number SEEDS of repetitions each with a different initial value
        for the parameters theta of the model is performed."""
    shape = get_theta_shape(model, layers)
    best_loss = float('inf')

    evolution = SeedStorage()
    similarity_measurement = model.qnode_generator(layers)

    for seed in range(seeds):
        key = jax.random.PRNGKey(seed)
        theta0 = jax.random.normal(key, shape)*SCALING_FACTOR
        training_result = train(similarity_measurement, theta0, hyperparameters_model, dataset, key)
        (evolution.val_loss).append(training_result.best_val_loss)
        (evolution.time).append(training_result.time)
        (evolution.epochs).append(training_result.epochs)

        if training_result.best_val_loss < best_loss:
            best_loss = training_result.best_val_loss
            best_result = training_result

    return TrainedModel(best_result, evolution)



def loss_v_epochs(training_result, layers: int, model_name: str):
    plt.rcParams.update({'font.size': 28})
    
    viridis = cm.get_cmap('viridis')
    color_train = viridis(0.2) 
    color_val = viridis(0.8)
    
    fig, ax1 = plt.subplots(figsize=(14, 8))
    epochs = list(range(1, training_result.epochs + 1))
    
    ax1.plot(epochs, training_result.val_losses,
             color=color_val, marker='o', linestyle='-', linewidth=2,
             label='Validation losses')
    ax1.plot(epochs, training_result.train_losses,
             color=color_train, marker='s', linestyle='--', linewidth=2,
             label='Train losses')
    
    ax1.margins(x=0.02, y=0.05)
    
    ax1.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
    

    ax1.set_xlabel('Epochs',fontsize=28)
    ax1.set_ylabel('Losses',fontsize=28)
    ax1.legend(loc='best', fontsize=28)
    ax1.grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(f'{model_name}-{layers}layers.png', bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig)



def train_and_evaluate_model(model, hyperparameters_model, layers, dataset, seeds):
    """Runs the training process over a dataset for a model with layers data reuploading layers and some hyperparameters.
        To perform an statistical evaluation of the model a number SEEDS of repetitions each with a different initial value
        for the parameters theta of the model is performed."""
    shape = get_theta_shape(model, layers)
    best_loss = float('inf')

    evolution = SeedStorage()
    similarity_measurement = model.qnode_generator(layers)

    for seed in range(seeds):
        key = jax.random.PRNGKey(seed)
        theta0 = jax.random.normal(key, shape)*SCALING_FACTOR
        training_result = train(similarity_measurement, theta0, hyperparameters_model, dataset, key)
        training_eval = evaluate_test(training_result, dataset, similarity_measurement)


        (evolution.val_loss).append(training_result.best_val_loss)
        (evolution.time).append(training_result.time)
        (evolution.accuracy).append(training_eval.accuracy)
        (evolution.auc).append(training_eval.auc)

        if training_result.best_val_loss < best_loss:
            best_loss = training_result.best_val_loss
            best_training_eval = training_eval
            best_training_result = training_result

    return EvaluatedModel(best_training_eval, evolution), best_training_result


def train(function, theta0, hyperparameters_model, dataset, key, shuffle=True) -> TrainingResult:
    """Training loop: trains the model by finding the values of the parameters theta that minimize the losses of the validation set
    returns the final parameters, losses of the validation set over the epochs, the total number of epochs needed and the time
    the loop over the epochs took."""
    theta = theta0

    @jax.jit
    def loss(theta, X, y):
        """Defines the loss function as the mean squared error"""
        def single_loss(x, label):
            pred = function(x, theta)
            return (pred - label) ** 2

        losses = jax.vmap(single_loss, in_axes=(0, 0))(X, y)
        return jnp.mean(losses)

    optimizer = optax.adam(hyperparameters_model.learning_rate)
    opt_state = optimizer.init(theta)

    @jax.jit
    def step(params, opt_state, X, y):
        """Single gradient descent step"""
        loss_val, grads = jax.value_and_grad(loss)(params, X, y)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss_val

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    best_theta = theta

    t_start = time.perf_counter()
    for epoch in range(hyperparameters_model.max_epochs):  # LOOP 1: over the epochs
        epoch_loss = 0.0

        key, subkey = jax.random.split(key)
        # shuffle the data so that the batches used
        # during each epoch are different
        perm = jax.random.permutation(subkey, dataset.len_train)
        X = dataset.train.features[perm]
        y = dataset.train.labels[perm]


        for start in range(0, dataset.len_train, hyperparameters_model.batch_size):  # LOOP 2: over the batches
            # for batch_size = 1: stochastic gradient descent
            # for batch_size = dataset.train.len_train: batch gradient descent
            # for any other value of batch_size in between: mini-batch gradient descent

            end = start + hyperparameters_model.batch_size
            batch_pairs = X[start:end]   # slices the training dataset into batches
            batch_labels = y[start:end]  # of size hyperparameters.batch_size
            # the loss function and the gradient will be computed over
            # one of these batches before updating the trainable parameters

            theta, opt_state, batch_loss = step(
                theta, opt_state, batch_pairs, batch_labels)

        train_loss = float(loss(theta, dataset.train.features, dataset.train.labels))
        train_losses.append(train_loss)
        val_loss = float(loss(theta, dataset.val.features, dataset.val.labels))
        val_losses.append(val_loss)
        epochs_used = epoch + 1
        # Early stopping check
        if val_loss < best_val_loss - hyperparameters_model.tolerance:  # continue
            best_val_loss = val_loss
            final_train_loss = train_loss
            best_theta = theta
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= hyperparameters_model.patience:
            break

    t_end = time.perf_counter()
    epochs_used = epoch + 1
    total_time = (t_end-t_start)

    return TrainingResult(best_theta, best_val_loss, final_train_loss, val_losses, train_losses, epochs_used, total_time)


def get_theta_shape(model: SQNNModel | QNNModel, layers: int) -> tuple:
    """Returns the correct shape for theta as a function of the chosen implementation

    For Tree Tensor circuits the last dimension is fixed to 2.
    Otherwise, it contains the number of rotations applied in the circuit"""
    if model.configuration.variational.form is VariationalForm.TREETENSOR:
        return (layers, model.configuration.qubits, 2)
    else:
        return (layers, model.configuration.qubits, model.configuration.variational.rotations)


def evaluate_test(training_result: TrainingResult, dataset: PairsSet, similarity_measurement, threshold=THRESHOLD) -> TrainingEvaluation:
    @jax.jit
    def predict(X):
        """Returns the set of predictions for all pairs of input vectors in X"""
        def single_predict(x):
            """Prediction over input pairs"""
            return similarity_measurement(x, training_result.theta)
        return jax.vmap(single_predict)(X)

    output = predict(dataset.test.features)
    y_pred = (output > threshold).astype(jnp.int64)

    accuracy = accuracy_score(y_true=dataset.test.labels, y_pred=y_pred)
    roc = roc_curve(y_true=dataset.test.labels, y_score=output)
    auc_score = roc_auc_score(y_true=dataset.test.labels, y_score=output)
    confusion = confusion_matrix(y_true=dataset.test.labels, y_pred=y_pred)
    return TrainingEvaluation(training_result,accuracy, roc, auc_score, confusion)

In [ ]:
from dataclasses import replace

class HyperparametersSweep:
    def __init__(self, hyperparameter_values):
        self.losses = {value: [] for value in hyperparameter_values}
        self.avg_losses = {value: [] for value in hyperparameter_values}
        self.std_losses = {value: [] for value in hyperparameter_values}
        self.time = {value: [] for value in hyperparameter_values}
        self.avg_time = {value: [] for value in hyperparameter_values}
        self.std_time = {value: [] for value in hyperparameter_values}
        self.avg_epochs = {value: [] for value in hyperparameter_values}
    
    def append_result(self, result: TrainedModel, hyperparameter_value):
        self.avg_time[hyperparameter_value].append(result.avg_time)
        self.time[hyperparameter_value].append(result.best.time)
        self.std_time[hyperparameter_value].append(result.std_time)
        self.losses[hyperparameter_value].append(result.best.best_val_loss)
        self.avg_losses[hyperparameter_value].append(result.avg_val_over_seeds)
        self.std_losses[hyperparameter_value].append(result.std_val_over_seeds)
        self.avg_epochs[hyperparameter_value].append(result.avg_epochs)
    

def run_layers(models, exp, hyperparameters_dict, dataset):
    """Trains and evaluates each model in models for a number layers in LAYERS_LIST of data reuploading layers"""
    for i in exp:
        print(f'model {i}')
        model = models[i]
        hyperparameters_model = hyperparameters_dict[i]
        for layers in LAYERS_LIST:
            result, _ = train_and_evaluate_model(model=model, hyperparameters_model=hyperparameters_model,layers=layers, dataset=dataset, seeds=SEEDS_EVALUATE)
            (model.result)[layers] = result
            print(f'L={layers}:  val loss {result.best.best_val_loss:.4f} |  train loss {result.best.final_train_loss:.4f} |  avg loss {np.average(result.val_over_seeds):.4f}  +- {np.std(result.val_over_seeds):.4f} '
                 f' | accuracy = {result.best.accuracy:.3f} | auc {result.best.auc:.3f} | epochs {result.best.epochs}' )
    return models


def hyperparameters_sweep_plot(architecture, sweep, values, xlabel, base, exp1):
    scale = base is not None
    model_labels = list(exp1)
    num_models = len(model_labels)

    plt.rcParams.update({'font.size':18})
    # Paleta viridis
    viridis = plt.get_cmap('viridis')
    colors = [viridis(i / max(1, num_models - 1)) for i in range(num_models+1)]
    # Sufijo según arquitectura
    arch_suffix = "similarity" if architecture is Architecture.SIMILARITY_FUNCTION else "overlap"

    fig3, ax = plt.subplots(figsize=(7, 5))
    for i in range(num_models):
        avg = np.array([sweep.avg_losses[item][i] for item in values])
        std = np.array([sweep.std_losses[item][i] for item in values])
        ax.plot(values, avg, marker='s', alpha=0.7,
                     color=colors[i], label=f'{model_labels[i]}', linewidth=4)
        ax.fill_between(values, avg - std, avg + std, color=colors[i], alpha=0.2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Average validation loss')
    if scale:
        ax.set_xscale('log', base=base)
    ax.grid(True, linestyle=':')
    plt.tight_layout()
    ax.legend(fontsize=13)
    plt.savefig(f'{xlabel}_avg_losses_{arch_suffix}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig3)

    fig4, ax = plt.subplots(figsize=(7, 5))
    for i in range(num_models):
        avg_t = np.array([sweep.avg_time[item][i] for item in values])
        std_t = np.array([sweep.std_time[item][i] for item in values])
        ax.plot(values, avg_t,  marker='s', alpha=0.7,
                    color=colors[i], label=f'{model_labels[i]}')
        ax.fill_between(values, avg_t - std_t, avg_t + std_t, color=colors[i], alpha=0.2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Average training time (s)')
    if scale:
        ax.set_xscale('log', base=base)
    ax.legend(fontsize=13)
    ax.grid(True, linestyle=':')
    plt.tight_layout()
    plt.savefig(f'{xlabel}_avg_time_{arch_suffix}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig4)

In [47]:
BATCH_SIZES = [2**i for i in range(int(np.log2(dataset.len_train))+1)]
BATCH_SIZES.append(dataset.len_train) # to consider batch gradient descent too
   
LEARNING_RATES = [1,5e-1,1e-1,5e-2,1e-2,5e-3,1e-3,5e-4,1e-4]

TOLERANCES = [1e-1, 1e-2, 1e-3, 1e-4,  1e-5, 1e-6]

PATIENCES = [1,5,10,15,20,25,30]

In [48]:
exp = tuple(key for key in configurations)

hyperparameters_similarity = {model:HyperparametersModel(max_epochs = 500, patience=10, tolerance = 1e-4, batch_size=32, learning_rate = 0.01) for model in exp}
models_sim = {i : SQNNModel(architecture=Architecture.SIMILARITY_FUNCTION, configuration=configurations[i]) for i in exp}
for i in exp:
    models_sim[i].hyperparameters = HyperparametersModel(max_epochs = 500, patience=10, tolerance = 1e-4, batch_size=32, learning_rate = 0.01) 

hyperparameters_overlap = {model:HyperparametersModel(max_epochs = 500, patience=10, tolerance = 1e-4, batch_size=32, learning_rate = 0.01) for model in exp}
models_overlap = {i : SQNNModel(Architecture.OVERLAP, configuration=configurations[i]) for i in exp}

original_dataset = sinus
dataset = sinus_pairs

In [ ]:
hyperparameters_overlap['angle-treetensor'] = HyperparametersModel(max_epochs = 500, patience=10, tolerance = 1e-4, batch_size=64, learning_rate = 0.5)
hyperparameters_overlap['angle-treetensor'] = HyperparametersModel(max_epochs = 500, patience=10, tolerance = 1e-4, batch_size=8, learning_rate = 0.05)

hyperparameters_overlap['angle-treetensor'] = HyperparametersModel(max_epochs = 500, patience=10, tolerance = 1e-4, batch_size=64, learning_rate = 0.05)
hyperparameters_overlap['angle-treetensor'] = HyperparametersModel(max_epochs = 500, patience=10, tolerance = 1e-4, batch_size=8, learning_rate = 0.01)

In [50]:
result = run_layers(models_overlap, exp, hyperparameters_overlap, dataset)

model angle-twolocal-1


AttributeError: 'tuple' object has no attribute 'best'